In [1]:
import pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1778160669937_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


#  Read CSV Files from S3

In [6]:
s3_path = "s3://vitaspark/retail_star_schema_csv/"

In [7]:
dim_store = spark.read.option("header", True).option("inferSchema", True).csv(s3_path + "dim_store.csv")

In [8]:
dim_product = spark.read.option("header", True).option("inferSchema", True).csv(s3_path + "dim_product.csv")

In [9]:
dim_customer = spark.read.option("header", True).option("inferSchema", True).csv(s3_path + "dim_customer.csv")

In [10]:
dim_time = spark.read.option("header", True).option("inferSchema", True).csv(s3_path + "dim_time.csv")

In [11]:
fact_sales = spark.read.option("header", True).option("inferSchema", True).csv(s3_path + "fact_sales.csv")

# Create Temporary Views

In [12]:
dim_store.createOrReplaceTempView("dim_store")

In [13]:
dim_product.createOrReplaceTempView("dim_product")

In [14]:
dim_customer.createOrReplaceTempView("dim_customer")

In [15]:
dim_time.createOrReplaceTempView("dim_time")

In [16]:
fact_sales.createOrReplaceTempView("fact_sales")

# Check All Tables

In [17]:
spark.sql("""
SELECT * FROM dim_store
""").show(truncate=False)

+--------+---------------------------+---------+-----------+------+
|store_id|store_name                 |city     |state      |region|
+--------+---------------------------+---------+-----------+------+
|1       |Mumbai Central Store       |Mumbai   |Maharashtra|West  |
|2       |Pune FC Road Store         |Pune     |Maharashtra|West  |
|3       |Bengaluru Indiranagar Store|Bengaluru|Karnataka  |South |
|4       |Delhi Connaught Store      |Delhi    |Delhi      |North |
|5       |Hyderabad Hitech Store     |Hyderabad|Telangana  |South |
+--------+---------------------------+---------+-----------+------+

In [18]:
spark.sql("""
SELECT * FROM dim_product
""").show(truncate=False)

+----------+--------------------+-----------+------------+----------+
|product_id|product_name        |category   |sub_category|unit_price|
+----------+--------------------+-----------+------------+----------+
|101       |Laptop Basic        |Electronics|Laptops     |45000.0   |
|102       |Smartphone X        |Electronics|Mobiles     |25000.0   |
|103       |Bluetooth Headphones|Electronics|Accessories |2500.0    |
|104       |Men T-Shirt         |Fashion    |Mens Wear   |799.0     |
|105       |Women Kurti         |Fashion    |Womens Wear |1299.0    |
|106       |Running Shoes       |Fashion    |Footwear    |2999.0    |
|107       |Rice 5kg            |Grocery    |Staples     |450.0     |
|108       |Cooking Oil 1L      |Grocery    |Kitchen     |180.0     |
|109       |Office Chair        |Furniture  |Chairs      |5500.0    |
|110       |Study Table         |Furniture  |Tables      |7500.0    |
+----------+--------------------+-----------+------------+----------+

In [19]:
spark.sql("""
SELECT * FROM dim_customer
""").show(truncate=False)

+-----------+-------------+------+---+---------+------------+
|customer_id|customer_name|gender|age|city     |loyalty_tier|
+-----------+-------------+------+---+---------+------------+
|1001       |Customer_01  |Male  |22 |Hyderabad|Silver      |
|1002       |Customer_02  |Male  |29 |Pune     |Regular     |
|1003       |Customer_03  |Female|23 |Mumbai   |Regular     |
|1004       |Customer_04  |Male  |35 |Mumbai   |Silver      |
|1005       |Customer_05  |Female|35 |Jaipur   |Gold        |
|1006       |Customer_06  |Male  |31 |Chennai  |Gold        |
|1007       |Customer_07  |Female|30 |Delhi    |Gold        |
|1008       |Customer_08  |Male  |26 |Chennai  |Regular     |
|1009       |Customer_09  |Female|43 |Hyderabad|Regular     |
|1010       |Customer_10  |Female|55 |Pune     |Platinum    |
|1011       |Customer_11  |Male  |39 |Nashik   |Silver      |
|1012       |Customer_12  |Male  |23 |Delhi    |Gold        |
|1013       |Customer_13  |Male  |35 |Pune     |Platinum    |
|1014   

In [20]:
spark.sql("""
SELECT * FROM dim_time
""").show(truncate=False)

+--------+-------------------+---+-----+----------+-------+----+
|date_key|full_date          |day|month|month_name|quarter|year|
+--------+-------------------+---+-----+----------+-------+----+
|20260101|2026-01-01 00:00:00|1  |1    |January   |1      |2026|
|20260102|2026-01-02 00:00:00|2  |1    |January   |1      |2026|
|20260103|2026-01-03 00:00:00|3  |1    |January   |1      |2026|
|20260104|2026-01-04 00:00:00|4  |1    |January   |1      |2026|
|20260105|2026-01-05 00:00:00|5  |1    |January   |1      |2026|
|20260106|2026-01-06 00:00:00|6  |1    |January   |1      |2026|
|20260107|2026-01-07 00:00:00|7  |1    |January   |1      |2026|
|20260108|2026-01-08 00:00:00|8  |1    |January   |1      |2026|
|20260109|2026-01-09 00:00:00|9  |1    |January   |1      |2026|
|20260110|2026-01-10 00:00:00|10 |1    |January   |1      |2026|
|20260111|2026-01-11 00:00:00|11 |1    |January   |1      |2026|
|20260112|2026-01-12 00:00:00|12 |1    |January   |1      |2026|
|20260113|2026-01-13 00:0

In [21]:
spark.sql("""
SELECT * FROM fact_sales
""").show(truncate=False)

+--------+--------+--------+----------+-----------+--------+----------+--------+------------+
|sales_id|date_key|store_id|product_id|customer_id|quantity|unit_price|discount|total_amount|
+--------+--------+--------+----------+-----------+--------+----------+--------+------------+
|1       |20260323|1       |101       |1009       |2       |45000.0   |0.05    |85500.0     |
|2       |20260118|1       |109       |1003       |5       |5500.0    |0.15    |23375.0     |
|3       |20260105|1       |102       |1007       |2       |25000.0   |0.0     |50000.0     |
|4       |20260313|2       |109       |1014       |2       |5500.0    |0.15    |9350.0      |
|5       |20260317|3       |101       |1006       |4       |45000.0   |0.1     |162000.0    |
|6       |20260205|2       |104       |1011       |1       |799.0     |0.0     |799.0       |
|7       |20260218|1       |106       |1012       |5       |2999.0    |0.1     |13495.5     |
|8       |20260414|1       |108       |1018       |1       |

# Aggregation Query 1: Total Sales by Store

In [22]:
spark.sql("""
SELECT 
    s.store_id,
    s.store_name,
    s.city,
    s.state,
    s.region,
    ROUND(SUM(f.total_amount), 2) AS total_sales
FROM fact_sales f
JOIN dim_store s
ON f.store_id = s.store_id
GROUP BY 
    s.store_id,
    s.store_name,
    s.city,
    s.state,
    s.region
ORDER BY total_sales DESC
""").show(truncate=False)

+--------+---------------------------+---------+-----------+------+-----------+
|store_id|store_name                 |city     |state      |region|total_sales|
+--------+---------------------------+---------+-----------+------+-----------+
|3       |Bengaluru Indiranagar Store|Bengaluru|Karnataka  |South |818869.2   |
|2       |Pune FC Road Store         |Pune     |Maharashtra|West  |756708.55  |
|1       |Mumbai Central Store       |Mumbai   |Maharashtra|West  |598869.95  |
|4       |Delhi Connaught Store      |Delhi    |Delhi      |North |408696.1   |
|5       |Hyderabad Hitech Store     |Hyderabad|Telangana  |South |349021.35  |
+--------+---------------------------+---------+-----------+------+-----------+

# Aggregation Query 2: Top 3 Product Categories by Total Sales

In [23]:
spark.sql("""
SELECT 
    p.category,
    ROUND(SUM(f.total_amount), 2) AS total_sales
FROM fact_sales f
JOIN dim_product p
ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
LIMIT 3
""").show(truncate=False)

+-----------+-----------+
|category   |total_sales|
+-----------+-----------+
|Electronics|2485750.0  |
|Furniture  |314500.0   |
|Fashion    |113100.65  |
+-----------+-----------+

# Aggregation Query 3: Average Sales Per Transaction for Each Month

In [24]:
spark.sql("""
SELECT 
    t.year,
    t.month,
    t.month_name,
    ROUND(AVG(f.total_amount), 2) AS average_sales_per_transaction
FROM fact_sales f
JOIN dim_time t
ON f.date_key = t.date_key
GROUP BY 
    t.year,
    t.month,
    t.month_name
ORDER BY 
    t.year,
    t.month
""").show(truncate=False)

+----+-----+----------+-----------------------------+
|year|month|month_name|average_sales_per_transaction|
+----+-----+----------+-----------------------------+
|2026|1    |January   |25905.59                     |
|2026|2    |February  |37888.35                     |
|2026|3    |March     |26336.4                      |
|2026|4    |April     |24248.59                     |
+----+-----+----------+-----------------------------+

# Aggregation Query 4: Customer with Highest Total Purchases

In [25]:
spark.sql("""
SELECT 
    c.customer_id,
    c.customer_name,
    c.gender,
    c.city,
    c.loyalty_tier,
    ROUND(SUM(f.total_amount), 2) AS total_purchase_amount
FROM fact_sales f
JOIN dim_customer c
ON f.customer_id = c.customer_id
GROUP BY 
    c.customer_id,
    c.customer_name,
    c.gender,
    c.city,
    c.loyalty_tier
ORDER BY total_purchase_amount DESC
LIMIT 1
""").show(truncate=False)

+-----------+-------------+------+---------+------------+---------------------+
|customer_id|customer_name|gender|city     |loyalty_tier|total_purchase_amount|
+-----------+-------------+------+---------+------------+---------------------+
|1019       |Customer_19  |Female|Hyderabad|Regular     |448657.5             |
+-----------+-------------+------+---------+------------+---------------------+